# Program Pengujian: Mengenali 1 Gambar

Alur: *upload* 1 gambar, baca berkas bobot, hitung umpan maju saja, lalu tampilkan apakah gambar dikenali atau tidak.
Mengikuti *Application procedure* Fausett, *Fundamentals of Neural Networks* (1994), hlm. 299 sampai 300.

Jalankan sel dari atas ke bawah. Sel 4 akan meminta Anda memilih 1 gambar.

## Sel 1. Pengaturan Lingkungan dan Ambang Batas

* Menghubungkan Google Drive, memuat daftar 3 nama kelas, serta menetapkan *threshold* (batas nilai minimal agar gambar dinyatakan dikenali) sebesar 0.5.

In [ ]:
import io                      # membaca berkas dari memori (bukan dari disk)
import os                      # akses folder dan berkas
import numpy as np             # operasi matriks dan vektor
from PIL import Image          # membaca gambar (Pillow)
import matplotlib.pyplot as plt  # menggambar grafik dan gambar
from google.colab import drive, files  # drive: sambung ke Google Drive; files: dialog unggah berkas

drive.mount('/content/drive')  # sambungkan Colab ke Google Drive

BERKAS_BOBOT = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI/BOBOT/BOBOT.npz'   # hasil dari program training
AMBANG_DIKENALI = 0.5    # threshold: nilai output terbesar minimal 0,5 agar gambar dianggap dikenali
NAMA_KELAS = ['PESAWAT', 'MOBIL', 'KAPAL']   # urutan sama dengan node output Y1, Y2, Y3

# Opsional: isi bila Anda tahu kelas gambar yang akan diunggah (0 = pesawat, 1 = mobil, 2 = kapal)
# agar hasil dibandingkan dengan kelas sebenarnya. Biarkan None bila tidak ingin dibandingkan.
KELAS_SEBENARNYA = None

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Berkas bobot    : {BERKAS_BOBOT}')
print(f'Ambang dikenali : {AMBANG_DIKENALI}')
print(f'Kelas           : {", ".join(NAMA_KELAS)}')

## Sel 2. Pemuatan Bobot Hasil Pelatihan

* Membaca berkas `BOBOT.npz` dari Drive (atau menyediakan dialog unggah berkas bila belum tersedia) dan memvalidasi kecocokan ukuran jaringan ($1024 \times 100 \times 3$).

In [ ]:
if os.path.exists(BERKAS_BOBOT):
    bobot = np.load(BERKAS_BOBOT)            # baca berkas .npz hasil training
    sumber = 'Google Drive'
else:
    print('Berkas bobot tidak ada di Drive. Silakan unggah BOBOT.npz.')
    unggahan = files.upload()                # files.upload: dialog Colab untuk memilih berkas dari komputer
    # list(unggahan.keys())[0]: nama berkas pertama yang diunggah; io.BytesIO: bungkus isinya agar bisa dibaca np.load
    bobot = np.load(io.BytesIO(unggahan[list(unggahan.keys())[0]]))
    sumber = 'unggahan'

v   = bobot['v']      # bobot V: dari node input ke node hidden
v_0 = bobot['v_0']    # bias node hidden
w   = bobot['w']      # bobot W: dari node hidden ke node output
w_0 = bobot['w_0']    # bias node output
LEBAR, TINGGI = [int(a) for a in bobot['ukuran']]   # ukuran gambar yang dipakai saat training

n = v.shape[0]   # node input: satu node untuk tiap pixel
p = v.shape[1]   # node hidden: lapisan tengah pengolah pola
m = w.shape[1]   # node output: satu per kelas

print('=' * 52)
print(f'BOBOT DIBACA (dari {sumber})')
print('=' * 52)
print(f'{"Variabel":<10}{"Ukuran":<14}')
for nama, larik in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)]:
    print(f'{nama:<10}{str(larik.shape):<14}')
print(f'\nNode input (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p): {p}')
print(f'Node output (m): {m}')
# assert: hentikan program bila ukuran bobot tidak sesuai (harus 3 node output dan n = lebar x tinggi)
assert m == 3 and n == LEBAR * TINGGI, 'Ukuran bobot tidak sesuai'

## Sel 3. Fungsi Prapemrosesan dan Umpan Maju

* Menyiapkan fungsi aktivasi sigmoid, fungsi `praproses` yang mengubah gambar menjadi vektor bernilai 0 sampai 1 (langkah sama persis dengan program *training*), serta fungsi `umpan_maju` (*feedforward*) satu arah tanpa koreksi bobot.

In [ ]:
def f(x):
    # fungsi aktivasi sigmoid: ubah nilai apa pun menjadi 0 sampai 1
    return 1.0 / (1.0 + np.exp(-x))


def praproses(gambar):
    """Ubah satu gambar PIL menjadi vektor x berisi n nilai, rentang 0 sampai 1 (sama dengan training)."""
    abu = gambar.convert('L')            # 1. grayscale: warna jadi satu nilai terang per pixel
    if abu.size != (LEBAR, TINGGI):      # 2. pengaman ukuran: bila ukuran beda,
        abu = abu.resize((LEBAR, TINGGI))   #    ubah ukuran (resize) ke ukuran training
    # 3. flatten: ubah gambar 2 dimensi jadi satu baris; 4. normalisasi: bagi 255 agar nilai 0 sampai 1
    return np.asarray(abu, dtype=np.float64).flatten() / 255.0


def umpan_maju(x):
    # umpan maju (feedforward): hitung keluaran y dari x, tanpa mengubah bobot
    z_in = v_0 + x @ v      # z_in: masukan bersih node hidden
    z    = f(z_in)          # z: keluaran node hidden
    y_in = w_0 + z @ w      # y_in: masukan bersih node output
    y    = f(y_in)          # y: keluaran node output
    return y

print('Fungsi siap: praproses(gambar) dan umpan_maju(x)')

## Sel 4. Pengunggahan Citra Uji

* Menampilkan tombol pemilih berkas untuk memasukkan satu gambar yang ingin diuji, lalu membaca ukuran dan mode warnanya (RGB = berwarna, L = abu-abu).

In [ ]:
unggahan = files.upload()                      # dialog Colab: pilih 1 gambar dari komputer
nama_berkas = list(unggahan.keys())[0]         # nama berkas pertama yang diunggah
if len(unggahan) > 1:
    print(f'Anda memilih {len(unggahan)} berkas. Yang dipakai hanya yang pertama: {nama_berkas}')

gambar = Image.open(io.BytesIO(unggahan[nama_berkas]))   # io.BytesIO: bungkus isi berkas agar bisa dibaca Pillow
print(f'Berkas   : {nama_berkas}')
print(f'Ukuran   : {gambar.size[0]} x {gambar.size[1]} pixel')
print(f'Mode     : {gambar.mode}')

## Sel 5. Klasifikasi dan Visualisasi Hasil

* Menjalankan citra lewat jaringan, memilih kelas dengan nilai keluaran tertinggi, lalu menentukan status pengenalan:
   * Jika nilai tertinggi $\ge 0.5$, citra DIKENALI.
   * Jika nilai tertinggi $< 0.5$, citra TIDAK DIKENALI.
* Menampilkan 3 panel: gambar asli, bentuk *grayscale* $32 \times 32$ yang diproses jaringan, serta grafik batang nilai keluaran ($Y_1, Y_2, Y_3$) beserta garis *threshold*.

In [ ]:
x = praproses(gambar)    # x: masukan jaringan (nilai pixel ternormalisasi)
y = umpan_maju(x)       # y: keluaran jaringan (Y1, Y2, Y3)

indeks = int(np.argmax(y))          # argmax: nomor node output bernilai terbesar = kelas tebakan
nilai_maks = float(y[indeks])
dikenali = nilai_maks >= AMBANG_DIKENALI   # True bila nilai terbesar mencapai threshold

print('=' * 52)
print('HASIL PENGENALAN')
print('=' * 52)
print(f'{"Node output":<14}{"Kelas":<10}{"Nilai":>8}')
for k in range(m):
    tanda = '  <-- terbesar' if k == indeks else ''
    print(f'Y{k + 1:<13}{NAMA_KELAS[k]:<10}{y[k]:>8.3f}{tanda}')
print('-' * 52)

if dikenali:
    print(f'DIKENALI sebagai {NAMA_KELAS[indeks]} (nilai {nilai_maks:.3f}, ambang {AMBANG_DIKENALI})')
else:
    print(f'TIDAK DIKENALI (nilai terbesar {nilai_maks:.3f}, di bawah ambang {AMBANG_DIKENALI})')

if KELAS_SEBENARNYA is not None:
    cocok = (indeks == KELAS_SEBENARNYA) and dikenali
    print(f'Kelas sebenarnya: {NAMA_KELAS[KELAS_SEBENARNYA]} -> {"BENAR" if cocok else "SALAH"}')

# 3 panel: gambar asli, gambar yang masuk jaringan, grafik batang nilai output
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(11, 3.2))
a1.imshow(gambar); a1.set_title('Gambar asli'); a1.axis('off')
a2.imshow(x.reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1); a2.set_title('Masuk ke jaringan (grayscale)'); a2.axis('off')
# batang hijau = node terbesar, abu-abu = lainnya; garis merah putus-putus = threshold
warna = ['tab:green' if k == indeks else 'tab:gray' for k in range(m)]
a3.bar([f'Y{k + 1}\n{NAMA_KELAS[k]}' for k in range(m)], y, color=warna)
a3.axhline(AMBANG_DIKENALI, color='tab:red', linestyle='--', linewidth=1)
a3.set_ylim(0, 1); a3.set_title('Nilai output')
plt.tight_layout()
plt.show()